In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os

load_dotenv()
LTA_KEY = os.getenv("LTA_KEY")

In [2]:
def get_bus_stops(account_key):
    """
    Requires LTA DataMall API key.
    """
    url = "https://datamall2.mytransport.sg/ltaodataservice/BusStops"
    headers = {"AccountKey": account_key, "accept": "application/json"}
    rows, skip = [], 0

    while True:
        r = requests.get(url, headers=headers, params={"$skip": skip}, timeout=30)
        r.raise_for_status()
        chunk = r.json().get("value", [])
        if not chunk:
            break
        rows.extend(chunk)
        skip += 500

    bus = pd.DataFrame(rows)
    # standardized columns
    if not bus.empty:
        bus = bus.rename(columns={
            "Latitude": "lat",
            "Longitude": "lon",
            "Description": "bus_stop_desc"
        })
    return bus

In [4]:
bus_df = get_bus_stops(account_key= LTA_KEY)
bus_df.head()

,BusStopCode,RoadName,bus_stop_desc,lat,lon
0,01012,Victoria St,Hotel Grand Pacific,1.296848,103.852536
1,01013,Victoria St,St. Joseph's Ch,1.297710,103.853225
2,01019,Victoria St,Bras Basah Cplx,1.296990,103.853022
3,01029,Nth Bridge Rd,Opp Natl Lib,1.296673,103.854414
4,01039,Nth Bridge Rd,Bugis Cube,1.298208,103.855491


In [5]:
bus_df.to_csv("../../data/cleaned/bus_stops_cleaned.csv", index=False)